In [0]:
from pyspark.sql import functions as F

### Create a big table containing 200 thousands rows

In [0]:
df_raw = spark.range(0,20000000) \
    .withColumn("groupBy_key",F.pmod(F.col("id"),F.lit(5))) \
    .withColumn("value",F.rand(seed=66))

In [0]:
df_raw.distinct().groupby("groupBy_key").count().show()

Analyzed the PySpark DataFrame operation that processes 20M in-memory rows with deduplication and grouping.

Summary

The query generates 20M rows in-memory, applies deduplication across all columns, then groups by a modulo key. The aggregation step consumes 124ms (77% of operator execution time) and 48MB of memory to produce 5 groups.

Details and metrics

The operation generates 20M unique IDs using Range, adds a grouping key (ID modulo 5) and random values
The deduplication step processes all 20M rows before aggregation; since the source contains unique IDs and random values with a fixed seed, duplicate rows are extremely unlikely
The aggregation phase reduces 20M rows down to 5 groups
Total execution time is 510ms (509ms compilation, 1ms execution); no external data is read
The .distinct() operation processes the full dataset but likely finds no duplicates to remove, given that spark.range() produces unique IDs. If you can confirm the dataset contains no duplicate rows (where id, groupBy_key, and value all match), removing .distinct() would eliminate unnecessary deduplication overhead and allow the groupby to process the data directly.


在大数据重工业管道中，“识别执行瓶颈”是区分普通开发人员与资深架构师的最高分水岭。推荐你在去翻看那些密密麻麻的耗时指标之前，先明白这件事在生产环境中的终极意义：

> **集群不是一尊完美的、匀速运转的机器。相反，分布式计算的常态是“混乱与不均”。**
> **一个拥有上百个节点的集群，可能 99% 的机器在跑了 2 秒钟后就已经收工在原地抽烟，而剩下的 1% 的机器却因为分到了脏数据、或者碰到了硬件死角，正在痛苦地发出 100% 满载的轰鸣，活生生把整个公司的核心业务管道卡死几个小时。**
> **识别瓶颈，就是要求你像一个经验老到的中医一样，在没有源码、只有冰冷图表和画像的情况下，一眼号出那 1% 的“病灶”在哪里，并用最省算力的工程手段给它“通血栓”。**
---

## 🧭 一、 调优第一核心认知：死盯“长木板”与“深色方块”

当你打开 Databricks 的 **Query Profile 瀑布图**，或者传统的 Spark UI 耗时瀑布图时，千万不要被上百个五颜六色的算子方块晃了眼。

* **工程铁律**：**大数据的整体耗时，永远取决于那个算得最慢的算子（木桶效应中的最长木板）。**
* **肉眼诊断法**：
在 Query Profile 中，Databricks 会极其聪明地把**耗时最长、卡住流水线时间最久**的那个物理算子，用**最深、最刺眼的颜色**或者**最大的方块面积**直接凸显出来。
* **架构师直觉**：
如果一个计划里有 50 个 `PhotonProject`（字段裁剪）和 `PhotonFilter`（过滤），它们每个只耗时 2 毫秒；而中间的 `PhotonShuffleExchangeSink`（网络洗牌下水道）卡了足足 **20 分钟**。你根本不需要去理会那 50 个小算子，把你全部的弹药和注意力死死盯住这个深色的 Shuffle 方块，它就是今天的头号刺客。

---

## 🛠️ 二、 大厂生产环境四大“终极瓶颈”与号脉特征

在工业界，99% 的性能大灾难，都可以归钱、归物、归因到以下四大重工业瓶颈中。我们按照你的“时间线与并行度”思维，把它们的号脉特征彻底理顺：

### 🚨 瓶颈 1：数据倾斜（Data Skew）—— 最大的“性能癌症”

* **幕后内幕**：分布式计算讲究“按 Key 分发”。如果你按“省份”分发数据，全中国 80% 的电商订单可能都在“广东”和“江浙沪”，剩下的 20% 分散在其他省份。结果负责“广东”的那个 Task 兵蚁直接被上亿条数据撑死（内存暴涨、耗时无穷无尽），而负责“西藏”的 Task 兵蚁两秒钟就去喝茶了。
* **Spark UI / Profile 号脉特征**：
拉开 Completed Tasks 的耗时分布表（Min, Median, 75th, Max）。如果你看到 **Median（中位数耗时）只有 2 秒**，而 **Max（最大耗时）却长达 1.5 小时**！这就是标准的数据倾斜晚期症状。全网机器都在陪着那一个被撑死的 Task 枯坐干等。
* **基础调优认知**：不能怪机器，是你的 Key 选得太垃圾。必须在代码里执行**加盐两阶段聚合（Salting）**，或者对大表提前进行拆分处理。

### 🚨 瓶颈 2：小文件灾难（Small File Problem）—— 隐形的“I/O 血栓”

* **幕后内幕**：上游分发了 2000 个 Task 并跑，结果每个 Task 极其死板地往分布式存储（HDFS/S3/ADLS）里写了一个只有 2KB 的碎文件，一共写了 2000 个。下游的任务去读这道数据时，CPU 为了在磁盘上高频打开、关闭这 2000 个小文件，光是握手开销就把算力烧光了。
* **Spark UI / Profile 号脉特征**：
在最底层的 Scan 节点（数据读取层），你发现它读取的 **Total Data Size（数据总大小）只有区区 10MB**，但是 **Task Total Time（算子总耗时）却诡异地消耗了几分钟**，且伴随着极其恐怖的磁盘 Metadata（元数据）查询耗时。
* **基础调优认知**：这就是你刚才在 Databricks 里看到的那个 **`Optimize` 按钮**存在的意义！必须立刻对上游执行 `OPTIMIZE` 合并小文件，或者在写入前调用 `.coalesce(1)` 强行把碎文件揉成 128MB 的健康大文件。

### 🚨 瓶颈 3：垃圾宽依赖（Unnecessary Shuffle）—— 冒烟的“虚拟网线”

* **幕后内幕**：这就是你刚才亲手揪出来的事故。在极其干净的全局唯一 Unique ID 流里，平白无故塞进一个 `.distinct()`，逼着全网机器无意义地大搬家、互扔文件。
* **Spark UI / Profile 号脉特征**：
在瀑布图中间，连续咬合着两个以上的 `Exchange / ShuffleExchange` 算子，并且 **Shuffle Write Bytes（写磁盘流量）** 和 **Shuffle Read Bytes（走网线流量）** 的数字大到和原始表一模一样。
* **基础调优认知**：
静态审查代码，**无情地砍掉无意义的 distinct()**；在发生大表 Join 小表时，反手甩出一行 `.broadcast()` 广播连接，直接把两阶段的大搬家（Shuffle Join）降维打击成零网络开销的原地计算（Broadcast Join）。

### 🚨 瓶颈 4：并行度过载或不足（Improper Partitions）—— 小兵的“分配不均”

* **幕后内幕**：面对 500GB 的滔天巨浪数据，你竟然极其吝啬地只给了 4 个默认物理分区（`splits=4`）。这意味着只有 4 个 Task 小兵去挑这 500GB 的大粪，每个小兵在内存里要啃上百个 G，直接当场把集群的内存撑爆（OOM 崩溃）。反之，面对 10 行数据，你却开辟了 200 个分区，200 个小兵去抢 10 行饭吃，光是线程调度的空转开销就超过了计算本身。
* **Spark UI / Profile 号脉特征**：
在 `PhotonRange` 或数据读取节点，死死盯住那个 **`splits` / `partitions**` 的初始数字。如果这个数字与你的实际数据量（Data Size）严重脱节，比例极其失调，它就是瓶颈。
* **基础调优认知**：大厂的工业黄金标准是：**确保每一个并跑的 Task 兵蚁，分到的内存数据量在 100MB 到 251MB 之间。** 太多就调大分区数（`spark.sql.shuffle.partitions`），太少就合并分区。

---

## Summary

```
🫀 大数据重工业管道：四大核心执行瓶颈号脉心法

1. 抓数据倾斜（性能癌症）
* **看表特征**：`Max Task Time` 远大于 `Median Task Time`（如 1.5小时 vs 2秒）。
* **调优直觉**：局部小兵被脏数据撑死，必须执行【加盐两阶段聚合】或大表打散。

2. 抓小文件灾难（I/O血栓）
* **看表特征**：数据总量极小（仅几MB），但底层 Scan 算子读取耗时诡异地长。
* **调优直觉**：零碎小文件逼疯了 CPU，必须立刻反手执行 Databricks 的【Optimize】或 `.coalesce()` 合并。

3. 抓垃圾宽依赖（网线冒烟）
* **看表特征**：瀑布图中密谋了多个深色 `Exchange` 算子，网络流量（Shuffle Read/Write）大到离谱。
* **调优直觉**：静态扫描，【砍掉冗余的distinct】或【开启大表Join小表的 broadcast 广播优化】。

4. 抓分区不合理（兵力失调）
* **看表特征**：底层的 `splits` 数字过大（小文件空转）或过小（单Task吞吐过载引发OOM）。
* **调优直觉**：微调并行度，死守【每个物理Task处理 100MB~200MB 数据】的工业黄金分割线。

```
